# Agentic BI System

## Autonomous Business Intelligence with Multi-Agent Collaboration

This system bypasses traditional BI dashboards by directly analyzing payment data and providing
actionable insights and recommendations like a senior consultant.

### Agent Team:
1. **Data Engineer Agent** - Reads and prepares data
2. **Data Scientist Agent** - Analyzes data and finds patterns
3. **Business Manager Agent** - Generates insights and recommendations
4. **Validator Agent** - Validates the quality of outputs

## 1. Setup: Import Libraries and Configure Environment

In [1]:
# =========================
# Imports
# =========================
import os
import json
from datetime import datetime
import openai

# Local modules
import tools
import utils

# =========================
# Environment & Client
# =========================
# Set your API key (use environment variable in production!)
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'  # Replace with your key

client = openai.OpenAI()

# Model configuration
MODEL = "gpt-4o-mini"  # Using GPT-4o-mini for cost efficiency

## 2. Company Context Definition

Define the business context that all agents will use to understand the company situation.

In [2]:
# =========================
# COMPANY CONTEXT
# =========================
COMPANY_CONTEXT = """
COMPANY: TechSaaS Pro - B2B SaaS Platform for Business Management

BUSINESS MODEL:
- Subscription-based SaaS platform operating in European markets
- Three subscription tiers: Basic, Pro, Enterprise
- Primary markets: Germany (DE), France (FR), Spain (ES), Italy (IT)
- Customer acquisition channels: Organic, Ads, Email marketing

CURRENT SITUATION (Q1 2026):
- Growing rapidly in the SMB segment
- Expanding enterprise customer base
- Testing new marketing channels effectiveness
- Optimizing payment processing with Stripe

KEY BUSINESS QUESTIONS:
1. Which customer segments drive the most revenue?
2. Which marketing channels are most effective?
3. Are there geographic patterns in customer behavior?
4. What is the health of our payment processing (success rates)?
5. Are there any anomalies or issues to investigate?

STRATEGIC PRIORITIES:
- Increase Enterprise segment revenue
- Optimize marketing spend by channel
- Expand in high-performing markets
- Maintain high payment success rates
"""

print("Company context loaded")

Company context loaded


## 3. System Architecture

Visualize how the agents work together.

In [3]:
# Display system architecture
utils.display_architecture_html()

## 4. Test Available Tools

Before running agents, let's verify our tools work correctly.

In [3]:
# Test: Get data summary
summary = tools.get_data_summary()
print(f"Dataset shape: {summary['shape']}")
print(f"Columns: {summary['columns'][:5]}...")  # First 5 columns

# Test: Calculate KPIs
kpis = tools.calculate_kpis()
print(f"\nTotal Revenue: EUR {kpis['kpis']['total_revenue_eur']:,.2f}")
print(f"Success Rate: {kpis['kpis']['success_rate']}%")

Dataset shape: {'rows': 696, 'columns': 23}
Columns: ['id', 'object', 'amount', 'amount_received', 'currency']...

Total Revenue: EUR 40,880.06
Success Rate: 100.0%


## 5. Agent Definitions

### 5.1 Data Engineer Agent
Reads and prepares data, identifies data quality issues, and provides data summary.

In [5]:
def data_engineer_agent(company_context: str = COMPANY_CONTEXT) -> dict:
    """
    Data Engineer Agent: Reads and prepares data for analysis.
    
    Returns:
        dict: Data summary and quality assessment
    """
    utils.log_agent_title_html("Data Engineer Agent", "DATA")
    
    prompt = f"""
You are a Data Engineer at a SaaS company.
Your role is to read, understand, and prepare data for analysis.

COMPANY CONTEXT:
{company_context}

YOUR TASK:
1. Read the payment data and understand its structure
2. Get a statistical summary of the dataset
3. Identify any data quality issues (missing values, anomalies)
4. Prepare a concise data profile for the Data Scientist

AVAILABLE TOOLS:
- read_csv_data: Read CSV data with optional sampling
- get_data_summary: Get statistical summary of the data

OUTPUT FORMAT (JSON):
{{
    "data_profile": {{
        "total_records": <number>,
        "columns": [<list of important columns>],
        "date_range": "<start> to <end>",
        "data_quality": "<good/moderate/poor>"
    }},
    "key_findings": [<list of data observations>],
    "recommendations_for_analysis": [<what to focus on>]
}}
"""
    
    messages = [{"role": "user", "content": prompt}]
    tools_list = tools.get_data_engineer_tools()
    
    # Agent loop
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_list,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        # If agent returns content, we're done
        if msg.content and not msg.tool_calls:
            utils.log_final_summary_html(msg.content)
            try:
                return json.loads(msg.content)
            except:
                return {"raw_output": msg.content}
        
        # Handle tool calls
        if msg.tool_calls:
            messages.append(msg)
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return {"error": "No tool calls or content returned"}

### 5.2 Data Scientist Agent
Performs analysis, calculates KPIs, and identifies patterns and anomalies.

In [6]:
def data_scientist_agent(data_profile: dict, company_context: str = COMPANY_CONTEXT) -> dict:
    """
    Data Scientist Agent: Analyzes data and finds patterns.
    
    Args:
        data_profile: Output from Data Engineer Agent
    
    Returns:
        dict: Analysis results with insights
    """
    utils.log_agent_title_html("Data Scientist Agent", "ANALYSIS")
    
    prompt = f"""
You are a Senior Data Scientist at a SaaS company.
Your role is to analyze payment data and find actionable insights.

COMPANY CONTEXT:
{company_context}

DATA PROFILE FROM DATA ENGINEER:
{json.dumps(data_profile, indent=2)}

YOUR TASK:
1. Calculate key performance indicators (KPIs)
2. Analyze revenue by segment, channel, and geography
3. Detect any anomalies in the data
4. Identify significant patterns and trends

AVAILABLE TOOLS:
- execute_analysis: Run predefined analyses (revenue_by_country, revenue_by_segment, revenue_by_channel, segment_channel_matrix)
- calculate_kpis: Get key performance indicators
- detect_anomalies: Find unusual transactions

OUTPUT FORMAT (JSON):
{{
    "kpis": {{
        "total_revenue": <number>,
        "avg_transaction": <number>,
        "success_rate": <percentage>
    }},
    "segment_analysis": {{<segment breakdown>}},
    "channel_analysis": {{<channel breakdown>}},
    "geographic_analysis": {{<country breakdown>}},
    "anomalies_found": <count>,
    "key_patterns": [<list of significant patterns>],
    "areas_of_concern": [<any issues to flag>]
}}
"""
    
    messages = [{"role": "user", "content": prompt}]
    tools_list = tools.get_data_scientist_tools()
    
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_list,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        if msg.content and not msg.tool_calls:
            utils.log_final_summary_html(msg.content)
            try:
                return json.loads(msg.content)
            except:
                return {"raw_output": msg.content}
        
        if msg.tool_calls:
            messages.append(msg)
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return {"error": "No tool calls or content returned"}

### 5.3 Business Manager Agent
Translates data insights into business recommendations and strategic actions.

In [7]:
def business_manager_agent(analysis_results: dict, company_context: str = COMPANY_CONTEXT) -> dict:
    """
    Business Manager Agent: Generates strategic insights and recommendations.
    
    Args:
        analysis_results: Output from Data Scientist Agent
    
    Returns:
        dict: Business insights and recommendations
    """
    utils.log_agent_title_html("Business Manager Agent", "BUSINESS")
    
    prompt = f"""
You are a Senior Business Consultant analyzing a SaaS company's performance. 
Think like a McKinsey consultant - provide actionable, strategic recommendations.

COMPANY CONTEXT:
{company_context}

DATA ANALYSIS RESULTS:
{json.dumps(analysis_results, indent=2)}

YOUR TASK:
1. Interpret the data analysis in business terms
2. Calculate additional KPIs if needed
3. Generate actionable business insights
4. Provide strategic recommendations with priority levels
5. Identify quick wins and long-term opportunities

AVAILABLE TOOLS:
- calculate_kpis: Get additional performance metrics
- generate_report_data: Get comprehensive report data

OUTPUT FORMAT (JSON):
{{
    "executive_summary": "<2-3 sentence overview>",
    "key_insights": [
        {{"insight": "<description>", "impact": "high/medium/low", "category": "revenue/growth/efficiency"}}
    ],
    "recommendations": [
        {{"action": "<specific action>", "priority": "immediate/short-term/long-term", "expected_impact": "<description>"}}
    ],
    "quick_wins": ["<list of easy improvements>"],
    "risks_to_monitor": ["<potential issues>"],
    "next_steps": ["<immediate actions>"]
}}
"""
    
    messages = [{"role": "user", "content": prompt}]
    tools_list = tools.get_business_manager_tools()
    
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_list,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        if msg.content and not msg.tool_calls:
            utils.log_final_summary_html(msg.content)
            try:
                return json.loads(msg.content)
            except:
                return {"raw_output": msg.content}
        
        if msg.tool_calls:
            messages.append(msg)
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return {"error": "No tool calls or content returned"}

### 5.4 Validator Agent
Validates the quality and consistency of agent outputs.

In [8]:
def validator_agent(business_report: dict, analysis_results: dict) -> dict:
    """
    Validator Agent: Validates quality and consistency of outputs.
    
    Args:
        business_report: Output from Business Manager Agent
        analysis_results: Output from Data Scientist Agent
    
    Returns:
        dict: Validation results
    """
    utils.log_agent_title_html("Validator Agent", "CHECK")
    
    prompt = f"""
You are a Quality Assurance Analyst.
Your role is to validate business reports for accuracy and completeness.

ANALYSIS RESULTS (Ground Truth):
{json.dumps(analysis_results, indent=2)}

BUSINESS REPORT TO VALIDATE:
{json.dumps(business_report, indent=2)}

VALIDATION CHECKS:
1. Are the insights supported by the data?
2. Are recommendations actionable and specific?
3. Is the executive summary accurate?
4. Are there any logical inconsistencies?
5. Are all key areas covered (revenue, segments, channels, geography)?

OUTPUT FORMAT (JSON):
{{
    "validation_status": "passed/failed/needs_review",
    "score": <0-100>,
    "checks": [
        {{"check": "<description>", "status": "passed/failed", "comment": "<optional>"}}
    ],
    "issues_found": ["<list of issues if any>"],
    "suggestions": ["<improvements if any>"]
}}
"""
    
    messages = [{"role": "user", "content": prompt}]
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    
    msg = response.choices[0].message
    
    if msg.content:
        try:
            result = json.loads(msg.content)
            status = result.get('validation_status', 'unknown')
            utils.log_validation_html(status, f"Score: {result.get('score', 'N/A')}/100")
            return result
        except:
            utils.log_final_summary_html(msg.content)
            return {"raw_output": msg.content}
    
    return {"error": "No validation output"}

## 6. Orchestrator: Run the Full Pipeline

Connect all agents together to run the complete analysis workflow.

In [9]:
def run_bi_pipeline(company_context: str = COMPANY_CONTEXT, validate: bool = True) -> dict:
    """
    Orchestrator: Runs the complete BI analysis pipeline.
    
    Args:
        company_context: Business context string
        validate: Whether to run validation agent
    
    Returns:
        dict: Complete analysis results
    """
    print("="*60)
    print("STARTING AGENTIC BI PIPELINE")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    results = {}
    
    # Step 1: Data Engineer
    print("\nStep 1/4: Data Engineering...")
    results['data_profile'] = data_engineer_agent(company_context)
    utils.log_handoff_html("Data Engineer", "Data Scientist", str(results['data_profile'])[:200])
    
    # Step 2: Data Scientist
    print("\nStep 2/4: Data Analysis...")
    results['analysis'] = data_scientist_agent(results['data_profile'], company_context)
    utils.log_handoff_html("Data Scientist", "Business Manager", str(results['analysis'])[:200])
    
    # Step 3: Business Manager
    print("\nStep 3/4: Business Insights...")
    results['business_report'] = business_manager_agent(results['analysis'], company_context)
    
    # Step 4: Validator (optional)
    if validate:
        print("\nStep 4/4: Validation...")
        results['validation'] = validator_agent(results['business_report'], results['analysis'])
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)
    
    return results

## 7. Execute the Pipeline

Run the complete analysis workflow. You can run individual agents or the full pipeline.

In [10]:
# Option A: Run individual agents for testing
# Uncomment the agent you want to test

# Test Data Engineer
# data_result = data_engineer_agent()
# print(json.dumps(data_result, indent=2))

In [10]:
# Option B: Run full pipeline
# This will execute all agents in sequence

# Uncomment to run:
final_results = run_bi_pipeline(validate=True)

STARTING AGENTIC BI PIPELINE
Time: 2026-03-31 17:25:21

Step 1/4: Data Engineering...



Step 2/4: Data Analysis...



Step 3/4: Business Insights...



Step 4/4: Validation...



PIPELINE COMPLETE


In [11]:
# Display final report (after running pipeline)
# Uncomment after running the pipeline:

if 'final_results' in dir() and final_results:
    utils.display_final_report_html({
        'summary': final_results.get('analysis', {}).get('kpis', {}),         'insights': final_results.get('business_report', {}).get('key_insights', []),
        'recommendations': final_results.get('business_report', {}).get('recommendations', [])
    })

## 8. Quick Test: Run Without API

Test the tools directly without needing the OpenAI API.

In [12]:
# Quick test of tools (no API needed)
print("Testing Tools Directly:\n")

# 1. Data Summary
print("1. Data Summary:")
summary = tools.get_data_summary()
print(f"   Rows: {summary['shape']['rows']}, Columns: {summary['shape']['columns']}")

# 2. KPIs
print("\n2. Key Performance Indicators:")
kpis = tools.calculate_kpis()
print(f"   Total Revenue: EUR {kpis['kpis']['total_revenue_eur']:,.2f}")
print(f"   Total Transactions: {kpis['kpis']['total_transactions']}")
print(f"   Success Rate: {kpis['kpis']['success_rate']}%")
print(f"   Avg Transaction: EUR {kpis['kpis']['average_transaction_value']:.2f}")

# 3. Revenue by Segment
print("\n3. Revenue by Segment:")
segment = tools.execute_analysis('revenue_by_segment')
for s in segment['data']:
    print(f"   {s['segment']}: EUR {s['total_revenue']:.2f} ({s['transaction_count']} txns)")

# 4. Revenue by Country
print("\n4. Revenue by Country:")
country = tools.execute_analysis('revenue_by_country')
for c in country['data']:
    print(f"   {c['country']}: EUR {c['total_revenue']:.2f}")

# 5. Anomalies
print("\n5. Anomaly Detection:")
anomalies = tools.detect_anomalies()
print(f"   Found {anomalies['anomaly_count']} anomalies")

Testing Tools Directly:

1. Data Summary:
   Rows: 696, Columns: 23

2. Key Performance Indicators:
   Total Revenue: EUR 40,880.06
   Total Transactions: 696
   Success Rate: 100.0%
   Avg Transaction: EUR 58.74

3. Revenue by Segment:
   basic: EUR 6539.56 (429 txns)
   enterprise: EUR 21493.07 (64 txns)
   pro: EUR 12847.43 (203 txns)

4. Revenue by Country:
   DE: EUR 13930.95
   ES: EUR 4684.81
   FR: EUR 18277.39
   IT: EUR 3986.91

5. Anomaly Detection:
   Found 41 anomalies
